# AI_REDACT — PII Redaction from Unstructured Text

## Use Case Overview

`AI_REDACT` detects and removes personally identifiable information (PII) from free-text fields using an LLM. Unlike regex-based redaction (which misses variations and context-dependent PII), `AI_REDACT` understands what constitutes sensitive information in context.

**SA use case:** Comply with GDPR, CCPA, and HIPAA requirements by automatically redacting PII before data is shared, analysed, or moved to lower-trust environments — all without writing custom parsing logic.

**Dataset:** 10 synthetic records representing support tickets, onboarding forms, incident reports, and legal documents — all containing various types of PII.

**PII types we'll redact:**
- `NAME` — personal names
- `EMAIL` — email addresses
- `PHONE` — phone numbers
- `SSN` — social security numbers
- `ADDRESS` — physical addresses
- `DATE_OF_BIRTH` — birth dates
- `CREDIT_CARD` — payment card numbers

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Preview Raw Data

In [ ]:
df = pd.read_sql("SELECT * FROM pii_records ORDER BY record_id", conn)
print(f"Records: {len(df)}")
df[["RECORD_ID","RECORD_TYPE","RAW_TEXT"]]

## Step 3 — Redact PII with AI_REDACT

`AI_REDACT(text, entity_types)` replaces detected PII with `[REDACTED_TYPE]` placeholders.

The `entity_types` array specifies which categories to redact — you can be selective (e.g. redact SSNs and credit cards but keep names for internal support use).

In [ ]:
redacted_df = pd.read_sql("""
    SELECT
        record_id,
        record_type,
        raw_text,
        AI_REDACT(
            raw_text,
            ['NAME', 'EMAIL', 'PHONE', 'SSN', 'ADDRESS', 'DATE_OF_BIRTH', 'CREDIT_CARD']
        ) AS redacted_text
    FROM pii_records
    ORDER BY record_id
""", conn)

for _, row in redacted_df.iterrows():
    print(f"\n--- Record {row['RECORD_ID']} ({row['RECORD_TYPE']}) ---")
    print(f"ORIGINAL: {row['RAW_TEXT']}")
    print(f"REDACTED: {row['REDACTED_TEXT']}")

## Step 4 — Selective Redaction by Record Type

Different record types have different compliance requirements. Redact only what is needed per context.

In [ ]:
selective_df = pd.read_sql("""
    SELECT
        record_id,
        record_type,
        CASE
            WHEN record_type IN ('medical_note', 'legal_document') THEN
                AI_REDACT(raw_text, ['NAME','SSN','DATE_OF_BIRTH','ADDRESS','EMAIL','PHONE','CREDIT_CARD'])
            WHEN record_type IN ('onboarding_form') THEN
                AI_REDACT(raw_text, ['SSN','CREDIT_CARD','DATE_OF_BIRTH'])
            ELSE
                AI_REDACT(raw_text, ['EMAIL','PHONE'])
        END AS redacted_text
    FROM pii_records
    ORDER BY record_id
""", conn)
selective_df[["RECORD_ID","RECORD_TYPE","REDACTED_TEXT"]]

## Step 5 — Create a Safe Analytics View

Create a view that always returns redacted text — lower-trust roles query this view and never see raw PII.

In [ ]:
conn.cursor().execute("""
    CREATE OR REPLACE VIEW pii_records_safe AS
    SELECT
        record_id,
        record_type,
        AI_REDACT(raw_text, ['NAME','EMAIL','PHONE','SSN','ADDRESS','DATE_OF_BIRTH','CREDIT_CARD']) AS text_content
    FROM pii_records
""")
print("Safe view created: pii_records_safe")
safe_view_df = pd.read_sql("SELECT * FROM pii_records_safe LIMIT 3", conn)
safe_view_df

## Step 6 — Interpretation & SA Tips

**AI_REDACT vs. regex-based redaction:**
- Regex misses context: `"my ID is 12345"` — regex won't know this is an account ID
- AI_REDACT understands semantic context: it redacts `"James Miller calling"` as a name even without a pattern

**SA tips:**
- Combine with Snowflake **Dynamic Data Masking** for role-based access: admins see raw text, analysts see redacted view
- Use `AI_REDACT` in a **Dynamic Table** to maintain a continuously-updated safe copy of incoming records
- For GDPR right-to-erasure compliance, `AI_REDACT` the data before sharing it with third parties
- Pair with `AI_EXTRACT` first to capture structured fields before redacting them from the source text
- Supported entity types include: `NAME`, `EMAIL`, `PHONE`, `SSN`, `ADDRESS`, `DATE_OF_BIRTH`, `CREDIT_CARD`, `IP_ADDRESS`, `PASSPORT`, `LICENSE_PLATE`